# Notebook 07: Neural Network Architecture Design

**Series 3: Advanced Methods & Neural Networks**  
**Team B: Advanced ML & Production Excellence**  
**Date**: 16/06/2025  
**Target**: Reproduce our **91.34% F1-Score Wide Network** architecture

---

## 🎯 **Learning Objectives**

1. **Reproduce Successful Architectures**: Implement our proven Wide and Deep network designs
2. **Architecture Analysis**: Compare wide vs deep network performance trade-offs
3. **Performance Excellence**: Achieve 91%+ F1-Score neural network foundations
4. **Ensemble Preparation**: Create models ready for ensemble integration

## 📊 **Reference Achievement**
- **Wide Network**: 91.34% F1-Score (our best architecture)
- **Training Efficiency**: <5 minutes training time
- **Memory Usage**: <2GB during training
- **Production Ready**: Optimized for inference speed

---


## 1. Environment Setup & Data Loading


In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

# Model persistence
import joblib
import time
from datetime import datetime
import json
import os

print("🚀 Team B: Neural Network Architecture Design - Environment Ready!")
print(f"📅 Timestamp: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")


🚀 Team B: Neural Network Architecture Design - Environment Ready!
📅 Timestamp: 16/06/2025 14:52:30


In [2]:
# Load training data
data_path = "../../data/SMSSPamCollection"
df = pd.read_csv(data_path, sep='\t', names=['label', 'message'])

print(f"📊 Dataset loaded: {len(df)} messages")
print(f"🏷️ Label distribution:")
print(df['label'].value_counts())
print(f"📈 Class balance: {df['label'].value_counts(normalize=True)}")

# Convert labels to binary
df['target'] = (df['label'] == 'spam').astype(int)
print(f"\n✅ Binary encoding: 0=ham, 1=spam")
print(f"Spam percentage: {df['target'].mean()*100:.2f}%")


📊 Dataset loaded: 5572 messages
🏷️ Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64
📈 Class balance: label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

✅ Binary encoding: 0=ham, 1=spam
Spam percentage: 13.41%


## 2. Feature Engineering & Preprocessing


In [3]:
# TF-IDF Vectorization (optimized for neural networks)
print("🔤 Creating TF-IDF features...")

# Optimized TF-IDF parameters for neural networks
vectorizer = TfidfVectorizer(
    max_features=5000,        # Moderate feature count for neural networks
    ngram_range=(1, 2),       # Unigrams and bigrams
    stop_words='english',     # Remove stop words
    min_df=2,                # Minimum document frequency
    max_df=0.95,             # Maximum document frequency
    sublinear_tf=True,       # Apply sublinear tf scaling
    norm='l2'                # L2 normalization
)

# Fit and transform
X_tfidf = vectorizer.fit_transform(df['message'])
y = df['target'].values

print(f"✅ TF-IDF matrix shape: {X_tfidf.shape}")
print(f"📊 Feature density: {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.4f}")
print(f"🎯 Target distribution: {np.bincount(y)}")

# Train-test split with stratification
print("\n📊 Creating train-test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"✅ Training set: {X_train.shape[0]} samples")
print(f"✅ Test set: {X_test.shape[0]} samples")
print(f"📈 Train spam ratio: {y_train.mean():.3f}")
print(f"📈 Test spam ratio: {y_test.mean():.3f}")

# Convert to dense arrays for neural networks
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

print(f"🧠 Dense matrices ready for neural networks")
print(f"💾 Memory usage: {X_train_dense.nbytes / 1024**2:.1f} MB (train)")


🔤 Creating TF-IDF features...
✅ TF-IDF matrix shape: (5572, 5000)
📊 Feature density: 0.0017
🎯 Target distribution: [4825  747]

📊 Creating train-test split...
✅ Training set: 4457 samples
✅ Test set: 1115 samples
📈 Train spam ratio: 0.134
📈 Test spam ratio: 0.134
🧠 Dense matrices ready for neural networks
💾 Memory usage: 170.0 MB (train)


## 3. Neural Network Architecture Implementation

### Reference: Our Successful Architectures
- **Wide Network**: (800, 400) - Our best performer (91.34% F1)
- **Deep Network**: (400, 200, 100, 50) - Alternative architecture
- **Optimized**: (512, 256, 128, 64) - Balanced approach


In [4]:
# Neural Network Architecture Definitions
architectures = {
    'wide_network': {
        'hidden_layer_sizes': (800, 400),
        'alpha': 0.001,
        'learning_rate_init': 0.001,
        'max_iter': 300,
        'early_stopping': True,
        'validation_fraction': 0.15,
        'n_iter_no_change': 20,
        'random_state': 42,
        'description': 'Wide Network - Our best performer (91.34% F1)'
    },
    
    'deep_network': {
        'hidden_layer_sizes': (400, 200, 100, 50),
        'alpha': 0.005,
        'learning_rate_init': 0.002,
        'max_iter': 400,
        'early_stopping': True,
        'validation_fraction': 0.15,
        'n_iter_no_change': 20,
        'random_state': 42,
        'description': 'Deep Network - Alternative architecture'
    },
    
    'balanced_network': {
        'hidden_layer_sizes': (512, 256, 128, 64),
        'alpha': 0.001,
        'learning_rate_init': 0.001,
        'max_iter': 350,
        'early_stopping': True,
        'validation_fraction': 0.15,
        'n_iter_no_change': 20,
        'random_state': 42,
        'description': 'Balanced Network - Optimized approach'
    }
}

print("🧠 Neural Network Architectures Defined:")
for name, config in architectures.items():
    print(f"  • {name}: {config['hidden_layer_sizes']} - {config['description']}")


🧠 Neural Network Architectures Defined:
  • wide_network: (800, 400) - Wide Network - Our best performer (91.34% F1)
  • deep_network: (400, 200, 100, 50) - Deep Network - Alternative architecture
  • balanced_network: (512, 256, 128, 64) - Balanced Network - Optimized approach


## 4. Training & Evaluation Framework


In [5]:
def train_and_evaluate_neural_network(name, config, X_train, y_train, X_test, y_test):
    """
    Train and evaluate a neural network with comprehensive metrics
    """
    print(f"\n🧠 Training {name}...")
    print(f"📋 Architecture: {config['hidden_layer_sizes']}")
    
    # Start timing
    start_time = time.time()
    
    # Create and train model
    model_config = {k: v for k, v in config.items() if k not in ['description']}
    model = MLPClassifier(**model_config)
    
    # Train with progress tracking
    model.fit(X_train, y_train)
    
    # Training time
    training_time = time.time() - start_time
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    # Results
    results = {
        'name': name,
        'architecture': config['hidden_layer_sizes'],
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'training_time': training_time,
        'n_iterations': model.n_iter_,
        'loss': model.loss_,
        'description': config['description']
    }
    
    # Print results
    print(f"✅ Training completed in {training_time:.1f} seconds")
    print(f"📊 Iterations: {model.n_iter_}")
    print(f"📈 F1-Score: {f1:.4f}")
    print(f"🎯 Precision: {precision:.4f}")
    print(f"🔍 Recall: {recall:.4f}")
    print(f"💰 Final Loss: {model.loss_:.6f}")
    
    return model, results

print("⚡ Training framework ready!")


⚡ Training framework ready!


## 5. Architecture Training & Comparison


In [6]:
# Train all architectures
models = {}
results = []

print("🚀 Starting Neural Network Architecture Training...")
print("=" * 80)

for name, config in architectures.items():
    model, result = train_and_evaluate_neural_network(
        name, config, X_train_dense, y_train, X_test_dense, y_test
    )
    models[name] = model
    results.append(result)
    print("-" * 80)

print("\n🎉 All architectures trained successfully!")


🚀 Starting Neural Network Architecture Training...

🧠 Training wide_network...
📋 Architecture: (800, 400)
✅ Training completed in 222.0 seconds
📊 Iterations: 47
📈 F1-Score: 0.9315
🎯 Precision: 0.9510
🔍 Recall: 0.9128
💰 Final Loss: 0.003135
--------------------------------------------------------------------------------

🧠 Training deep_network...
📋 Architecture: (400, 200, 100, 50)
✅ Training completed in 99.2 seconds
📊 Iterations: 23
📈 F1-Score: 0.9209
🎯 Precision: 0.9922
🔍 Recall: 0.8591
💰 Final Loss: 0.004772
--------------------------------------------------------------------------------

🧠 Training balanced_network...
📋 Architecture: (512, 256, 128, 64)
✅ Training completed in 107.2 seconds
📊 Iterations: 28
📈 F1-Score: 0.9352
🎯 Precision: 0.9514
🔍 Recall: 0.9195
💰 Final Loss: 0.003215
--------------------------------------------------------------------------------

🎉 All architectures trained successfully!


In [7]:
# Results comparison
results_df = pd.DataFrame(results)

print("📊 Neural Network Architecture Comparison:")
print("=" * 80)
print(results_df[['name', 'f1_score', 'precision', 'recall', 'training_time']].round(4))

# Find best performer
best_model_name = results_df.loc[results_df['f1_score'].idxmax(), 'name']
best_f1 = results_df['f1_score'].max()

print(f"\n🏆 Best Architecture: {best_model_name}")
print(f"📈 Best F1-Score: {best_f1:.4f}")

# Check if we achieved our target
target_f1 = 0.91
if best_f1 >= target_f1:
    print(f"✅ TARGET ACHIEVED! {best_f1:.4f} >= {target_f1:.2f}")
else:
    print(f"⚠️ Target not reached: {best_f1:.4f} < {target_f1:.2f}")


📊 Neural Network Architecture Comparison:
               name  f1_score  precision  recall  training_time
0      wide_network    0.9315     0.9510  0.9128       222.0184
1      deep_network    0.9209     0.9922  0.8591        99.2275
2  balanced_network    0.9352     0.9514  0.9195       107.2046

🏆 Best Architecture: balanced_network
📈 Best F1-Score: 0.9352
✅ TARGET ACHIEVED! 0.9352 >= 0.91


## 6. Model Persistence & Ensemble Preparation


In [8]:
# Save best neural network model
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
model_dir = '../../models/neural_networks_series3/'
os.makedirs(model_dir, exist_ok=True)

# Save best model
best_model_path = f"{model_dir}neural_network_{best_model_name}_{timestamp}.joblib"
joblib.dump(models[best_model_name], best_model_path)

# Save vectorizer
vectorizer_path = f"{model_dir}tfidf_vectorizer_{timestamp}.joblib"
joblib.dump(vectorizer, vectorizer_path)

# Save all models for ensemble
ensemble_models = {}
for name, model in models.items():
    model_path = f"{model_dir}neural_network_{name}_{timestamp}.joblib"
    joblib.dump(model, model_path)
    ensemble_models[name] = {
        'path': model_path,
        'f1_score': results_df[results_df['name'] == name]['f1_score'].iloc[0],
        'architecture': str(architectures[name]['hidden_layer_sizes'])
    }

print(f"💾 Models saved to: {model_dir}")
print(f"🏆 Best model: {best_model_path}")
print(f"🔤 Vectorizer: {vectorizer_path}")
print(f"🤝 Ensemble models: {len(ensemble_models)} architectures ready")


💾 Models saved to: ../../models/neural_networks_series3/
🏆 Best model: ../../models/neural_networks_series3/neural_network_balanced_network_16062025_145939.joblib
🔤 Vectorizer: ../../models/neural_networks_series3/tfidf_vectorizer_16062025_145939.joblib
🤝 Ensemble models: 3 architectures ready


## 7. Summary & Next Steps


In [9]:
print("🎯 NOTEBOOK 07: NEURAL NETWORK ARCHITECTURE DESIGN - COMPLETE!")
print("=" * 80)

print(f"🏆 ACHIEVEMENTS:")
print(f"  • Best Architecture: {best_model_name} ({architectures[best_model_name]['hidden_layer_sizes']})")
print(f"  • F1-Score: {best_f1:.4f} ({'✅ TARGET ACHIEVED' if best_f1 >= 0.91 else '⚠️ TARGET MISSED'})")
print(f"  • Training Time: {results_df[results_df['name'] == best_model_name]['training_time'].iloc[0]:.1f} seconds")

print(f"\n💾 DELIVERABLES:")
print(f"  • Neural network models: {len(models)} architectures trained")
print(f"  • Best model saved: {best_model_path}")
print(f"  • Ensemble-ready models: {len(ensemble_models)} variants")
print(f"  • TF-IDF vectorizer: {vectorizer_path}")

print(f"\n🚀 NEXT STEPS (Notebook 08):")
print(f"  • Advanced training optimization")
print(f"  • Hyperparameter tuning with GridSearch")
print(f"  • Early stopping and validation monitoring")
print(f"  • Target: Optimize to 91%+ F1-Score")

print(f"\n📈 STATUS: Ready for Notebook 08 - Advanced Model Training & Optimization")
print("=" * 80)


🎯 NOTEBOOK 07: NEURAL NETWORK ARCHITECTURE DESIGN - COMPLETE!
🏆 ACHIEVEMENTS:
  • Best Architecture: balanced_network ((512, 256, 128, 64))
  • F1-Score: 0.9352 (✅ TARGET ACHIEVED)
  • Training Time: 107.2 seconds

💾 DELIVERABLES:
  • Neural network models: 3 architectures trained
  • Best model saved: ../../models/neural_networks_series3/neural_network_balanced_network_16062025_145939.joblib
  • Ensemble-ready models: 3 variants
  • TF-IDF vectorizer: ../../models/neural_networks_series3/tfidf_vectorizer_16062025_145939.joblib

🚀 NEXT STEPS (Notebook 08):
  • Advanced training optimization
  • Hyperparameter tuning with GridSearch
  • Early stopping and validation monitoring
  • Target: Optimize to 91%+ F1-Score

📈 STATUS: Ready for Notebook 08 - Advanced Model Training & Optimization
